# Plantilla: clasificación multiclase (agnóstica al dataset)

Notebook **base** para predecir una variable con **K clases** (K ≥ 3; especies, tipos, categorías…). No está ligado a un dataset concreto.

## Pasos principales (ejecutar en orden)

| Paso | Sección | Qué haces |
|------|---------|-----------|
| **0** | Helpers | Imports y funciones compartidas de preprocesado y modelos. |
| **1** | Explorar CSV | `PREVIEW_PATH`, `PREVIEW_SEP` — columnas, tipos, faltantes, conteo por clase. |
| **2** | CONFIG | Rutas, `RAW_LABEL_COL`, `CLASS_NAMES` (opcional); define `build_models(n_classes)`. |
| **3** | Carga | Leer el CSV. |
| **4** | Calidad de datos | Número de clases y faltantes. |
| **5** | Visualización | Barras por clase (y scatter si hay features numéricas). |
| **6** | Split | **X**, **y**; train / val / test (`split_train_val_test`, estratificado). |
| **7** | Preprocesado | Numéricas → imputer + escalar; categóricas → imputer + one-hot. |
| **8** | Comparar modelos | Entrena en **train**, mide en **val**; orden por **accuracy** (F1 weighted en tabla). |
| **9** | Mejor modelo | Elige en val; reentrena train+val; reporte y matriz en **test**. |

Tras la carga, el target queda en **0..K-1** (`prepare_multiclass_target`); **K** puede ser cualquier entero ≥ 3. `CLASS_NAMES=None` infiere nombres del CSV; `build_models(N_CLASSES)` se ejecuta en el paso 3.

Preprocesado en `Pipeline` (`fit` en train). Fechas, ids o texto libre: excluir con `DROP_COLS` o convertir antes; si no encajan en numéricas/categóricas, se descartan.

### Primera vez con tu CSV

1. Copia el archivo a `data/`.
2. Paso **1** — comprueba **K ≥ 3** clases en el target.
3. Paso **2** (CONFIG).
4. Pasos **3**–**9** en orden.

**Ejemplos ya resueltos:** `03-clasificacion-multiple-iris.ipynb` · `03-clasificacion-multiple-wine.ipynb` · `03-clasificacion-multiple-thyroid.ipynb` (binario: `02-clasificacion-binaria-thyroid.ipynb`)

> Ejecuta Jupyter desde `07.b-ejemplos-supervisados/`.



In [ ]:
# =============================================================================
# Helpers — funciones reutilizables (misma lógica en todo el benchmark)
# =============================================================================
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer  # pipeline distinto por tipo de columna
from sklearn.impute import SimpleImputer  # rellenar NaN antes de escalar/codificar
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline  # encadena: preprocesado → modelo
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")


def infer_feature_columns(df, target_col, drop_cols, feature_cols):
    """Lista de columnas predictoras (X).

    - Si FEATURE_COLS no es None: usa esa lista explícita.
    - Si no: todas las columnas excepto TARGET_COL y DROP_COLS (ids, leakage…).
    """
    if feature_cols is not None:
        return list(feature_cols)
    exclude = {target_col, *drop_cols}
    return [c for c in df.columns if c not in exclude]


def infer_column_types(X, numeric_cols=None, categorical_cols=None):
    """Separa columnas para ColumnTransformer.

    Por defecto: int/float → numéricas; object/category/bool/string → categóricas.
    Si solo defines NUMERIC_COLS, las categóricas siguen infiriéndose (y viceversa).
    Para control total, define ambas listas en CONFIG.
    """
    if numeric_cols is None:
        numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    if categorical_cols is None:
        categorical_cols = X.select_dtypes(
            include=["object", "category", "bool", "string"]
        ).columns.tolist()
    return list(numeric_cols), list(categorical_cols)


def build_preprocess(numeric_cols, categorical_cols):
    """Preprocesador único compartido por todos los modelos del benchmark.

    Numéricas: imputar (mediana) → StandardScaler.
    Categóricas (texto): imputar (moda) → OneHotEncoder.
    El .fit() ocurre dentro de pipe.fit(X_train) — en test solo .transform() (sin leakage).
    Columnas de X no listadas aquí se descartan (remainder='drop').
    """
    transformers = []

    if numeric_cols:
        transformers.append(
            (
                "num",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_cols,
            )
        )

    if categorical_cols:
        transformers.append(
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "encoder",
                            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                        ),
                    ]
                ),
                categorical_cols,
            )
        )

    if not transformers:
        raise ValueError("No hay columnas numéricas ni categóricas para preprocesar.")
    return ColumnTransformer(transformers, remainder="drop")


def prepare_multiclass_target(df, *, raw_label_col, target_col, class_names=None):
    """Codifica el target a enteros 0..K-1. class_names=None infiere K del CSV."""
    out = df.copy()
    if raw_label_col is not None:
        series = out[raw_label_col]
        if class_names is None:
            class_names = sorted(series.dropna().unique(), key=lambda x: str(x))
        label_to_id = {label: i for i, label in enumerate(class_names)}
        out[target_col] = series.map(label_to_id)
        if out[target_col].isna().any():
            bad = series[out[target_col].isna()].unique()
            raise ValueError(f"Etiquetas sin mapear en CLASS_NAMES: {bad!r}")
    else:
        out[target_col] = pd.to_numeric(out[target_col], errors="raise")
        values = sorted(out[target_col].dropna().unique())
        k = len(values)
        if class_names is None:
            class_names = [str(int(v)) if float(v).is_integer() else str(v) for v in values]
        elif len(class_names) != k:
            raise ValueError(
                f"CLASS_NAMES ({len(class_names)}) no coincide con {k} clases: {values}"
            )
        ints = [int(v) for v in values]
        if ints != list(range(k)):
            raise ValueError(
                f"TARGET_COL debe ser 0..{k - 1} sin huecos; encontrado: {ints}"
            )
    out[target_col] = out[target_col].astype(int)
    k = len(class_names)
    if k < 3:
        raise ValueError(
            f"Multiclase requiere >= 3 clases (K={k}). Usa plantilla binaria si K=2."
        )
    return out, list(class_names)


def split_train_val_test(X, y, test_size, val_size, random_state, stratify=False):
    """Divide en train, validación y test (dos llamadas a train_test_split).

    - test_size: fracción del total para test (hold-out final, paso 9).
    - val_size: fracción de train+val; el benchmark (paso 8) usa solo val.
    Con test_size=0.2 y val_size=0.25 → ~60 % train, ~20 % val, ~20 % test.
    """
    kw = dict(test_size=test_size, random_state=random_state)
    if stratify:
        kw["stratify"] = y
    X_tv, X_test, y_tv, y_test = train_test_split(X, y, **kw)
    kw2 = dict(test_size=val_size, random_state=random_state)
    if stratify:
        kw2["stratify"] = y_tv
    X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, **kw2)
    return X_train, X_val, X_test, y_train, y_val, y_test



## 1. Explorar el CSV (antes de CONFIG)

Pon la ruta de **tu** archivo. Si todo aparece en una columna, cambia `PREVIEW_SEP`.

In [ ]:
# --- Paso 1: explorar SIN tocar CONFIG todavía ---
PREVIEW_PATH = "data/mi_dataset.csv"  # ruta a tu CSV
PREVIEW_SEP = ","  # separador: ","  |  ";"  |  "\t"

# Carga provisional solo para inspeccionar estructura
df_preview = pd.read_csv(PREVIEW_PATH, sep=PREVIEW_SEP)

print(f"Filas: {len(df_preview):,}  |  Columnas: {len(df_preview.columns)}")
print("\n--- Nombres de columnas (índice : nombre) ---")
for i, col in enumerate(df_preview.columns):
    print(f"  {i:2d}: {col!r}")

print("\n--- Tipos de datos (dtypes) ---")
print(df_preview.dtypes)

print("\n--- Primeras filas ---")
display(df_preview.head())

# Faltantes: el pipeline imputará después; aquí solo diagnosticamos
print("\n--- Valores faltantes por columna ---")
missing = df_preview.isna().sum()
if missing.any():
    display(missing[missing > 0].to_frame("nulos"))
else:
    print("No hay valores faltantes.")

# Ayuda para rellenar NUMERIC_COLS / CATEGORICAL_COLS en CONFIG
_num = df_preview.select_dtypes(include=[np.number]).columns.tolist()
_cat = df_preview.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()
print("\n--- Sugerencia automática de tipos ---")
print("Numéricas (int/float):", _num)
print("Categóricas (object/category/bool/string):", _cat)
print(
    "\n>>> Siguiente: en CONFIG pon DATA_PATH, CSV_SEP iguales y elige TARGET_COL "
    "(K clases; CLASS_NAMES en CONFIG o None para inferir)."
)



## 2. CONFIG — adaptar a tu dataset

Copia los valores de la exploración. **Solo esta sección** cambia entre proyectos.


In [ ]:
# ========== Paso 2: CONFIG — único bloque que cambia entre datasets ==========
DATA_PATH = 'data/mi_dataset.csv'
CSV_SEP = ','

# XGBoost y CatBoost exigen target numérico (0..K-1). Si el CSV trae texto, define RAW_LABEL_COL.
RAW_LABEL_COL = "nombre_columna_texto"  # None si TARGET_COL ya es entero en el CSV
TARGET_COL = "target"
# None = inferir K y nombres del CSV (paso 3); o lista explícita [nombre_0, …, nombre_{K-1}]
CLASS_NAMES = None
DROP_COLS = ['nombre_columna_texto']
FEATURE_COLS = None
NUMERIC_COLS = None
CATEGORICAL_COLS = None

TEST_SIZE = 0.2
VAL_SIZE = 0.25
RANDOM_STATE = 42
METRIC_PRINCIPAL = 'accuracy'

def build_models(n_classes):
    """Diccionario nombre → estimador. Comenta líneas para excluir modelos del benchmark."""
    from sklearn.ensemble import (
        GradientBoostingClassifier,
        HistGradientBoostingClassifier,
        RandomForestClassifier,
    )
    from sklearn.linear_model import LogisticRegression
    from sklearn.neighbors import KNeighborsClassifier
    from xgboost import XGBClassifier
    from catboost import CatBoostClassifier

    if n_classes < 3:
        raise ValueError(f"build_models: K={n_classes} < 3 (multiclase)")
    models = {
        "LogisticRegression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "RandomForest": RandomForestClassifier(
            n_estimators=100,
            criterion="gini",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features="sqrt",
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            bootstrap=True,
            oob_score=False,
            max_samples=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "HistGradientBoosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(
            random_state=RANDOM_STATE,
            verbosity=0,
            n_estimators=100,
            objective="multi:softmax",
            num_class=n_classes,
            n_jobs=-1,
        ),
        "CatBoost": CatBoostClassifier(
            random_state=RANDOM_STATE,
            verbose=False,
            iterations=100,
            allow_writing_files=False,
            loss_function="MultiClass",
        ),
    }
    return models






## 3. Carga de datos


In [ ]:
# --- Paso 3: carga definitiva con los parámetros de CONFIG ---
df = pd.read_csv(DATA_PATH, sep=CSV_SEP)

df, CLASS_NAMES = prepare_multiclass_target(
    df,
    raw_label_col=RAW_LABEL_COL,
    target_col=TARGET_COL,
    class_names=CLASS_NAMES,
)
N_CLASSES = len(CLASS_NAMES)
MODELS = build_models(N_CLASSES)

print("Shape:", df.shape)
print(f"Clases (K={N_CLASSES}):", CLASS_NAMES)
print("\nDistribución del target:")
print(
    df[TARGET_COL]
    .value_counts()
    .sort_index()
    .rename(index=lambda i: CLASS_NAMES[i])
)
df.head()


## 4. Calidad de datos


In [ ]:
# --- Paso 4: balance de clases y faltantes ---
print(
    df[TARGET_COL]
    .value_counts()
    .sort_index()
    .rename(index=lambda i: CLASS_NAMES[i])
)
print("\nFaltantes:")
print(df.isna().sum().pipe(lambda s: s[s > 0] if s.any() else "Sin faltantes"))




## 5. Visualización rápida


In [ ]:
# --- Paso 5: conteo por clase ---
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET_COL].value_counts().plot(kind="bar", ax=ax)
ax.set_title("Distribución de clases")
ax.set_xlabel(TARGET_COL)
plt.tight_layout()
plt.show()



## 6. X / y y split (estratificado)


In [ ]:
# --- Paso 6: separar features (X), target (y) y dividir train / test ---
feature_cols = infer_feature_columns(df, TARGET_COL, DROP_COLS, FEATURE_COLS)
X = df[feature_cols]
y = df[TARGET_COL]

# Clasificación de columnas para el ColumnTransformer
numeric_cols, categorical_cols = infer_column_types(X, NUMERIC_COLS, CATEGORICAL_COLS)

print("Numéricas:", len(numeric_cols), "| Categóricas:", len(categorical_cols))


X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y,
    test_size=TEST_SIZE,
    val_size=VAL_SIZE,
    random_state=RANDOM_STATE,
    stratify=True,
)
print(
    f"Tamaños → train: {len(X_train):,} | val: {len(X_val):,} | test: {len(X_test):,}"
)



## 7. Preprocesado


In [ ]:
# --- Paso 7: definir el preprocesador (mismo objeto para todos los modelos) ---
preprocess = build_preprocess(numeric_cols, categorical_cols)
preprocess  # muestra la estructura: ramas num y cat



## 8. Comparar modelos

Métricas en **train** y **val**; brecha train−val para detectar **overfitting**.



In [ ]:
# --- Paso 8: métricas y benchmark de modelos ---

OVERFIT_GAP_WARN = 0.15  # brecha train-val en METRIC_PRINCIPAL por encima → posible overfitting

def classification_metrics(y_true, y_pred):
    """Accuracy, precision, recall y F1 (weighted en multiclase)."""
    from sklearn.metrics import (
        accuracy_score,
        f1_score,
        precision_score,
        recall_score,
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


def evaluate_models(models, preprocess, X_train, X_val, y_train, y_val):
    """Entrena en train; métricas en train y val para detectar overfitting."""
    rows = []
    metric = METRIC_PRINCIPAL
    for name, estimator in models.items():
        pipe = Pipeline([("preprocess", preprocess), ("model", estimator)])
        pipe.fit(X_train, y_train)
        m_train = classification_metrics(y_train, pipe.predict(X_train))
        m_val = classification_metrics(y_val, pipe.predict(X_val))
        row = {"modelo": name}
        for k, v in m_train.items():
            row[f"{k}_train"] = v
        for k, v in m_val.items():
            row[f"{k}_val"] = v
        row[f"gap_{metric}"] = m_train[metric] - m_val[metric]
        rows.append(row)
    return pd.DataFrame(rows).sort_values(f"{metric}_val", ascending=False)


results = evaluate_models(MODELS, preprocess, X_train, X_val, y_train, y_val)
display(results.round(4))

gap_col = f"gap_{METRIC_PRINCIPAL}"
sospechosos = results[results[gap_col] > OVERFIT_GAP_WARN]
if len(sospechosos):
    print(
        f"\nPosible overfitting (gap {METRIC_PRINCIPAL} train-val > {OVERFIT_GAP_WARN}):"
    )
    display(
        sospechosos[
            ["modelo", f"{METRIC_PRINCIPAL}_train", f"{METRIC_PRINCIPAL}_val", gap_col]
        ].round(4)
    )
else:
    print(
        f"\nSin gap {METRIC_PRINCIPAL} train-val > {OVERFIT_GAP_WARN} "
        "(no hay señal fuerte de overfitting en el benchmark)."
    )

fig, ax = plt.subplots(figsize=(9, 5))
plot_df = results.set_index("modelo")[
    [f"{METRIC_PRINCIPAL}_train", f"{METRIC_PRINCIPAL}_val"]
]
plot_df.plot(kind="barh", ax=ax)
ax.set_xlabel(METRIC_PRINCIPAL)
ax.set_title("Train vs validación — clasificación multiclase")
ax.legend(["train", "val"])
plt.tight_layout()
plt.show()





## 9. Detalle del mejor modelo

Matriz de **confusión** e informe por clase en **test** (no aplica a regresión; allí se usa correlación).


In [ ]:
# --- Paso 9: reporte y matriz de confusión del mejor modelo ---
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

best_name = results.iloc[0]["modelo"]
print(f"Mejor modelo (val): {{best_name}}\n")

X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])

best_pipe = Pipeline([("preprocess", preprocess), ("model", MODELS[best_name])])
best_pipe.fit(X_trainval, y_trainval)
y_pred = best_pipe.predict(X_test)

print("Métricas en test (tras reentrenar con train+val):")
display(pd.DataFrame([classification_metrics(y_test, y_pred)]).round(4))
print()

print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=CLASS_NAMES
)
plt.title(f"Matriz de confusión (test) — {{best_name}}")
plt.tight_layout()
plt.show()




## Checklist: nuevo dataset (multiclase)

1. CSV en `data/` → explorar → CONFIG.
2. `CLASS_NAMES=None` o lista de K nombres; target 0..K-1 tras paso 3 (`N_CLASSES`).
3. `DROP_COLS` incluye `RAW_LABEL_COL` si codificas desde texto.
4. Split **estratificado** por `y`.
5. Métrica principal: **accuracy** o **F1 weighted**.
